# xxxx

## 1. Data Structure and Basic Characteristics 

In [38]:
import pandas as pd
import numpy as np

df = pd.read_json("../data/raw_credit_applications.json")
df.head()

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData types:")
print(df.dtypes)

Shape: (502, 8)

Columns:
Index(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision',
       'processing_timestamp', 'loan_purpose', 'notes'],
      dtype='object')

Data types:
_id                     object
applicant_info          object
financials              object
spending_behavior       object
decision                object
processing_timestamp    object
loan_purpose            object
notes                   object
dtype: object


### Observations: 

The dataset contains 502 credit applications and 8 top-level fields.

The structure is semi-structured (nested JSON), meaning several fields contain embedded objects and arrays. This increases the risk of structural inconsistencies and hidden data quality issues.

All columns are currently loaded as type "object", meaning further normalization is required for proper auditing and validation.

# 2. Data Quality Assessment 

We assess the dataset using the six data quality dimensions defined in the course:

1. Completeness  
2. Uniqueness  
3. Validity  
4. Consistency  
5. Timeliness  
6. Accuracy  

## 2.1 Completeness

Completeness evaluates whether required data fields are present and populated.

Given the nested JSON structure, completeness must be assessed both at:
- Top-level fields
- Nested object fields (applicant_info, financials, decision)

In [5]:
df.isnull().sum()

_id                       0
applicant_info            0
financials                0
spending_behavior         0
decision                  0
processing_timestamp    440
loan_purpose            452
notes                   500
dtype: int64

In [6]:
applicant_df = pd.json_normalize(df["applicant_info"])
financials_df = pd.json_normalize(df["financials"])
decision_df = pd.json_normalize(df["decision"])

In [7]:
print("Applicant missing:")
print(applicant_df.isnull().sum())

print("\nFinancials missing:")
print(financials_df.isnull().sum())

print("\nDecision missing:")
print(decision_df.isnull().sum())

Applicant missing:
full_name        0
email            0
ssn              5
ip_address       5
gender           1
date_of_birth    1
zip_code         1
dtype: int64

Financials missing:
annual_income              5
credit_history_months      0
debt_to_income             0
savings_balance            0
annual_salary            497
dtype: int64

Decision missing:
loan_approved         0
rejection_reason    292
interest_rate       210
approved_amount     210
dtype: int64


In [8]:
total = len(df)

print("Top-level missing percentage:")
print((df.isnull().sum() / total * 100).round(2))

print("\nApplicant missing percentage:")
print((applicant_df.isnull().sum() / total * 100).round(2))

print("\nFinancials missing percentage:")
print((financials_df.isnull().sum() / total * 100).round(2))

print("\nDecision missing percentage:")
print((decision_df.isnull().sum() / total * 100).round(2))

Top-level missing percentage:
_id                      0.00
applicant_info           0.00
financials               0.00
spending_behavior        0.00
decision                 0.00
processing_timestamp    87.65
loan_purpose            90.04
notes                   99.60
dtype: float64

Applicant missing percentage:
full_name        0.0
email            0.0
ssn              1.0
ip_address       1.0
gender           0.2
date_of_birth    0.2
zip_code         0.2
dtype: float64

Financials missing percentage:
annual_income             1.0
credit_history_months     0.0
debt_to_income            0.0
savings_balance           0.0
annual_salary            99.0
dtype: float64

Decision missing percentage:
loan_approved        0.00
rejection_reason    58.17
interest_rate       41.83
approved_amount     41.83
dtype: float64


### Findings – Completeness

Completeness was assessed at both the top-level and nested-object level due to the semi-structured JSON design of the dataset.

At the top level, three attributes show critical missingness:
notes (99.6%), loan_purpose (90.0%), and processing_timestamp (87.7%).
The high absence of processing_timestamp raises governance and auditability concerns, as timestamps are typically required in financial systems.

Within nested objects, applicant and core financial attributes are largely complete (below 1% missing in most cases). However, annual_salary is missing in nearly all records (98.9%), indicating a likely schema inconsistency rather than random data loss.

Decision-related attributes (rejection_reason, interest_rate, approved_amount) display conditional missingness, as their presence depends on loan approval status. This suggests the need for cross-field validation rather than direct completeness correction.

Overall, completeness risks are concentrated in metadata fields and schema design rather than in primary applicant data.

## 2.2 Uniqueness

Uniqueness evaluates whether records that should be distinct are not duplicated.

In [9]:
print("Duplicate _id count:")
print(df["_id"].duplicated().sum())

Duplicate _id count:
2


In [12]:
print("Duplicate applicant SSN:")
print(applicant_df["ssn"].duplicated().sum())

Duplicate applicant SSN:
7


In [13]:
print("Duplicate email:")
print(applicant_df["email"].duplicated().sum())

Duplicate email:
8


In [ ]:
df[df["_id"].duplicated(keep=False)].  #duplicates

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
8,app_042,"{'full_name': 'Joseph Lopez', 'email': 'joseph...","{'annual_income': 69000, 'credit_history_month...","[{'category': 'Insurance', 'amount': 153}, {'c...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
354,app_042,"{'full_name': 'Joseph Lopez', 'email': 'joseph...","{'annual_income': 69000, 'credit_history_month...","[{'category': 'Insurance', 'amount': 153}, {'c...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,RESUBMISSION
383,app_001,"{'full_name': 'Stephanie Nguyen', 'email': 'st...","{'annual_income': 102000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 576}]","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
455,app_001,"{'full_name': 'Stephanie Nguyen', 'email': 'st...","{'annual_income': 102000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 576}]","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,DUPLICATE_ENTRY_ERROR


### Findings – Uniqueness

Uniqueness was assessed by verifying that identifiers expected to be unique do not contain duplicates.

Two duplicate _id values were detected. Since _id represents the application identifier, duplicates indicate a structural data integrity issue in the ingestion or storage pipeline.

Seven duplicate SSNs and eight duplicate emails were also identified. While these may reflect repeat applications by the same individual, they may also indicate potential fraud risk or insufficient identity validation controls.

Overall, uniqueness issues are primarily structural at the application ID level and require remediation at the data ingestion stage.

## 2.3 Validity

Validity evaluates whether data values fall within acceptable and logical ranges.

### 2.3.1  Financial Field Validation

In [15]:
financials_df.dtypes

annual_income             object
credit_history_months      int64
debt_to_income           float64
savings_balance            int64
annual_salary            float64
dtype: object

Before performing rule-based validation checks, data types were verified. The annual_income field was stored as an object type, requiring conversion to numeric format.

In [17]:
financials_df["annual_income"] = pd.to_numeric(
    financials_df["annual_income"], 
    errors="coerce"
)

In [18]:
financials_df.dtypes

annual_income            float64
credit_history_months      int64
debt_to_income           float64
savings_balance            int64
annual_salary            float64
dtype: object

In [19]:
print("Negative annual income:")
print((financials_df["annual_income"] < 0).sum())

print("\nNegative savings balance:")
print((financials_df["savings_balance"] < 0).sum())

print("\nDebt-to-income > 1:")
print((financials_df["debt_to_income"] > 1).sum())

print("\nNegative credit history months:")
print((financials_df["credit_history_months"] < 0).sum())

Negative annual income:
0

Negative savings balance:
1

Debt-to-income > 1:
1

Negative credit history months:
2


### Findings - Financial Fields Validation

Four financial validity issues were identified:

- 1 record contains a negative savings balance.
- 1 record has a debt-to-income ratio greater than 1.
- 2 records contain negative credit history months.

Negative credit history values are logically impossible and represent clear data entry or ingestion errors.

The debt-to-income ratio exceeding 1 suggests either scaling inconsistencies or calculation errors.

These findings indicate minor but structurally important financial data quality issues.

### 2.3.2 Decision Field Validation

Decision-level validity checks were performed to ensure that financial outcome variables fall within logical numeric ranges. Specifically, interest rates and approved amounts were tested for negative values.

In [22]:
print("Negative interest rate:")
print((decision_df["interest_rate"] < 0).sum())

print("\nNegative approved amount:")
print((decision_df["approved_amount"] < 0).sum())

Negative interest rate:
0

Negative approved amount:
0


#### Findings - Decisions Field Validation

No negative values were identified in either `interest_rate` or `approved_amount`, indicating valid numeric ranges for decision-related monetary fields.

### 2.3.3 Cross-field logical validation

Beyond individual field validation, logical consistency between decision attributes was assessed.

Certain fields are conditionally dependent on the loan approval status:

- If `loan_approved` is True, both `interest_rate` and `approved_amount` should be populated.
- If `loan_approved` is False, `rejection_reason` should be populated.
- Financial terms should not be present for rejected applications.

These checks ensure business-rule consistency within the decision data.

In [23]:
print("Approved but missing interest rate:")
print(((decision_df["loan_approved"] == True) & 
       (decision_df["interest_rate"].isnull())).sum())

print("\nApproved but missing approved amount:")
print(((decision_df["loan_approved"] == True) & 
       (decision_df["approved_amount"].isnull())).sum())

print("\nRejected but missing rejection reason:")
print(((decision_df["loan_approved"] == False) & 
       (decision_df["rejection_reason"].isnull())).sum())

print("\nRejected but interest rate present:")
print(((decision_df["loan_approved"] == False) & 
       (decision_df["interest_rate"].notnull())).sum())

Approved but missing interest rate:
0

Approved but missing approved amount:
0

Rejected but missing rejection reason:
0

Rejected but interest rate present:
0


### Findings – Cross-Field Logical Validation

No logical inconsistencies were identified between loan approval status and dependent decision fields.

All approved applications contain valid interest rates and approved amounts.  
All rejected applications contain appropriate rejection reasons and no financial terms.

This indicates strong internal consistency within the decision-making data structure.

## 2.4 Consistency 

Consistency evaluates whether data representations are uniform across records and aligned with expected schema definitions.

### 2.4.1 Schema Consistency – Financial Fields

In [24]:
print("Financial fields:")
print(financials_df.columns)

Financial fields:
Index(['annual_income', 'credit_history_months', 'debt_to_income',
       'savings_balance', 'annual_salary'],
      dtype='object')


In [25]:
financials_df[["annual_income", "annual_salary"]].head()

,annual_income,annual_salary
0,73000.0,NaN
1,78000.0,NaN
2,61000.0,NaN
3,103000.0,NaN
4,57000.0,NaN


In [26]:
salary_overlap = financials_df[
    financials_df["annual_salary"].notnull()
][["annual_income", "annual_salary"]]

salary_overlap

,annual_income,annual_salary
76,NaN,45000.0
94,NaN,46000.0
99,NaN,94000.0
141,NaN,86000.0
149,NaN,75000.0


In [27]:
print("Mismatched values:")
print((salary_overlap["annual_income"] != salary_overlap["annual_salary"]).sum())

Mismatched values:
5


### Findings – Schema Consistency (Financial Fields)

A structural schema inconsistency was identified in the financial data.

Two fields representing the same business concept (`annual_income` and `annual_salary`) coexist in the dataset. However, they are not used simultaneously.

Records populated with `annual_salary` contain null values in `annual_income`, and vice versa. This indicates schema fragmentation rather than simple duplication.

Such inconsistency suggests incomplete schema standardization during data ingestion and requires consolidation into a single authoritative field.

Suggestion: This inconsistency should be resolved through schema consolidation and standardization into a single authoritative field.

### 2.4.2 Representation Consistency – Decision Fields

In [28]:
print("Unique values in loan_approved:")
print(decision_df["loan_approved"].unique())

Unique values in loan_approved:
[False  True]


The `loan_approved` field is consistently encoded as a boolean variable (True/False) across all records. No categorical representation inconsistencies were identified.

Overall, consistency issues in the dataset are structural rather than representational.

## 2.5 Timeliness

Timeliness evaluates whether data is recorded and updated within an acceptable time frame 

### 2.5.1 Processing Timestamp Availability

In [29]:
print("Missing processing timestamps:")
print(df["processing_timestamp"].isnull().sum())
print("Percentage missing:")
print((df["processing_timestamp"].isnull().mean() * 100).round(2))

Missing processing timestamps:
440
Percentage missing:
87.65


#### Findings – Processing Timestamp Availability

A significant timeliness issue was identified in the dataset.

Out of 502 records, 440 (87.65%) do not contain a `processing_timestamp`. The absence of processing time metadata prevents assessment of data freshness, processing latency, and operational monitoring.

In financial systems, timestamp tracking is essential for auditability, SLA monitoring, and regulatory compliance. The high proportion of missing timestamps indicates a major limitation in temporal traceability.

Suggestion: Timestamp capture should be enforced at ingestion time to ensure full processing traceability.

### 2.5.2 Timestamp Format & Type Validation

In [30]:
print(df["processing_timestamp"].dtype)

object


In [31]:
df["processing_timestamp_parsed"] = pd.to_datetime(
    df["processing_timestamp"],
    errors="coerce",
    utc=True
)

print("Parsing failures:")
print(df["processing_timestamp_parsed"].isnull().sum())

Parsing failures:
440


In [32]:
print(df["processing_timestamp_parsed"].dt.tz)

UTC


### Findings – Timestamp Format & Type Validation

The `processing_timestamp` field was originally stored as an object type, requiring explicit conversion to datetime format.

After parsing, no malformed timestamp values were detected beyond the 440 previously identified missing records. This indicates that available timestamps follow a consistent format.

Parsed timestamps are timezone-aware (UTC), which aligns with best practices for distributed and financial systems.

However, despite correct formatting in existing records, the high proportion of missing timestamps (87.65%) severely limits temporal traceability and operational monitoring capabilities.

## 2.6 Accuracy

Accuracy evaluates whether data correctly represents real-world values.

In the absence of external reference datasets, accuracy was approximated through internal plausibility and logical consistency checks.

In [33]:
print("Max annual income:", financials_df["annual_income"].max())
print("Min annual income:", financials_df["annual_income"].min())

print("Max savings balance:", financials_df["savings_balance"].max())
print("Min savings balance:", financials_df["savings_balance"].min())

Max annual income: 171000.0
Min annual income: 0.0
Max savings balance: 88078
Min savings balance: -5000


## 2.7 Findings – Plausibility Assessment

Observed financial values fall within economically plausible ranges.

Annual income values range from 0 to 171,000, which is consistent with realistic income distributions. Savings balances range from -5,000 to 88,078, suggesting the presence of overdraft scenarios rather than impossible values.

No extreme or clearly unrealistic financial figures were identified.

However, in the absence of external reference data, true accuracy cannot be fully validated. The assessment therefore relies on internal plausibility checks rather than factual verification.

### Data Quality Issues by Dimension

| Dimension    | Issue Identified |
|--------------|------------------|
| Completeness | Missing processing_timestamp |
| Consistency  | annual_income vs annual_salary schema conflict |
| Validity     | Negative credit history months |
| Accuracy     | Debt-to-income > 1 potential scaling issue |

## 3. Remediation Strategy & Cleaning Pipeline

Following the identification and quantification of data quality issues, this section demonstrates concrete remediation steps implemented to improve structural integrity, logical validity, and analytical usability of the dataset.

All transformations are reproducible and form a structured data cleaning pipeline.

### 3.1 Schema Standardization (Income Fields)

In [34]:
# Consolidate income fields into a single authoritative column
financials_df["annual_income_clean"] = (
    financials_df["annual_income"]
    .fillna(financials_df["annual_salary"])
)

# Drop redundant column
financials_df.drop(columns=["annual_salary"], inplace=True)

### 3.2 Invalid Value Handling

In [39]:
# Negative credit history months → invalid
financials_df.loc[
    financials_df["credit_history_months"] < 0,
    "credit_history_months"
] = np.nan

# Debt-to-income > 1 → potential scaling error
financials_df.loc[
    financials_df["debt_to_income"] > 1,
    "debt_to_income"
] = np.nan

# Negative savings (optional business rule decision)
financials_df.loc[
    financials_df["savings_balance"] < 0,
    "savings_balance"
] = np.nan

### 3.3 Duplicate Handling

In [40]:
# Remove duplicate application IDs
df = df.drop_duplicates(subset="_id")

### 3.4 Timestamp Normalization

In [41]:
df["processing_timestamp"] = pd.to_datetime(
    df["processing_timestamp"],
    errors="coerce",
    utc=True
)

## 4. Clean Dataset Summary

In [42]:
print("Remaining missing values:")
print(df.isnull().sum())

print("\nFinal dataset shape:")
print(df.shape)

Remaining missing values:
_id                       0
applicant_info            0
financials                0
spending_behavior         0
decision                  0
processing_timestamp    438
loan_purpose            450
notes                   500
dtype: int64

Final dataset shape:
(500, 8)


Certain fields (loan_purpose, notes) were intentionally left unchanged as they are optional metadata and not structurally required for credit risk modeling.

In [45]:
def clean_dataset(df):
    # all cleaning steps
    return df_clean